# Experiment B2: warm-start continuation as plain LoRA

**Тезис:** адаптивные методы (AdaLoRA, L1RA) нужны только в первые ~25% обучения для
установления per-module rank-pattern. Остальные ~75% можно дотренировать как plain LoRA.

**Алгоритм:**
1. Загружаем `warm_start_snapshot/` из baseline_v2 (4 прогона: meta_math/open_orca × adalora/l1ra).
2. Бинарным поиском по |E| (AdaLoRA) или |C| (L1RA) находим глобальный порог τ такой,
   что среднее число выживших индексов на модуль = `TARGET_K = 16`.
3. Поглощаем gate в A: AdaLoRA `A' = E ⊙ A`; L1RA `A' = C ⊙ A` + транспонирование под PEFT.
4. Per-module прунинг по survivor mask. Модули с K=0 исключаются из target_modules
   (нет адаптера → метод сам решил, что здесь адаптация не нужна).
5. Собираем plain-LoRA с `rank_pattern={module: K_m}`. Total trainable params матчится с
   baseline LoRA r=16: `sum(K_m) ≈ 196 × 16 = 3136` индексов.
6. Переносим AdamW state (m1, m2) для survivor индексов; scheduler и RNG из snapshot.
7. Дотренировываем на оставшиеся `2500 - trigger_step` оптимайзер-шагов в plain-LoRA режиме.

## 0. Colab: mount Drive (локально пропустить)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/diploma/project

## 1. Imports & seed

In [ ]:
import sys
sys.path.insert(0, '.')

import gc
import json
import os
import random
from functools import partial

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src import (
    SFTDataset, collate_fn, filter_df_by_max_len,
    build_adalora_model, build_l1ra_model,
    build_optimizer, build_scheduler, train_model,
)
from src.warm_start_load import warm_start_to_plain_lora


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

## 2. Config

In [ ]:
SEED         = 42
MODEL_NAME   = 'Qwen/Qwen3-0.6B'
MAX_LENGTH   = 1024
BATCH_SIZE   = 8
GRAD_ACCUM   = 4
NUM_EPOCHS   = 2
LR           = 1e-4
TARGET_K     = 16
TRAIN_SIZE   = 40_000
VAL_SIZE     = 4_000
DATA_DIR     = './data'
BASELINE_RUNS_DIR = './logs_v2'
B2_RUNS_DIR       = './logs_v2_B2'
DATASETS_SELECTED = ['meta_math', 'open_orca']
SAVE_STEPS   = 50
LOG_PER_STEP = True

os.makedirs(B2_RUNS_DIR, exist_ok=True)
print('B2 output dir:', os.path.abspath(B2_RUNS_DIR))

## 3. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 4. Data preparation

Используем те же 50k-parquet'ы что в `baseline_v2.ipynb` (датасеты должны быть уже скачаны).

In [ ]:
N_RAW = 50_000

meta_math = (
    pd.read_parquet(f'{DATA_DIR}/meta-math_MetaMathQA_{N_RAW}.parquet')
    .rename(columns={'query': 'question', 'response': 'answer'})
    [['question', 'answer']]
)
open_orca = (
    pd.read_parquet(f'{DATA_DIR}/Open-Orca_OpenOrca_{N_RAW}.parquet')
    [['question', 'response']]
    .rename(columns={'response': 'answer'})
)
print(f'raw: meta_math={len(meta_math)}  open_orca={len(open_orca)}')

meta_math_filtered, _ = filter_df_by_max_len(meta_math, tokenizer, MAX_LENGTH)
open_orca_filtered, _ = filter_df_by_max_len(open_orca, tokenizer, MAX_LENGTH)

def split_train_val(df, train_size=TRAIN_SIZE, val_size=VAL_SIZE, name=''):
    need = train_size + val_size
    assert len(df) >= need, f'[{name}] {len(df)} < {need}'
    return df.iloc[:train_size].reset_index(drop=True), df.iloc[train_size:train_size+val_size].reset_index(drop=True)

_filtered = {'meta_math': meta_math_filtered, 'open_orca': open_orca_filtered}
_splits = {n: split_train_val(_filtered[n], name=n) for n in DATASETS_SELECTED}
for n, (tr, va) in _splits.items():
    print(f'{n:20s}  train={len(tr)}  val={len(va)}')

DATASETS = _splits

## 5. DataLoader factory

In [ ]:
def make_loaders(train_df, val_df, seed: int = SEED):
    train_ds = SFTDataset(train_df, tokenizer, max_length=MAX_LENGTH)
    val_ds   = SFTDataset(val_df,   tokenizer, max_length=MAX_LENGTH)
    _collate = partial(collate_fn, tokenizer=tokenizer)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=_collate, generator=g)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=_collate)
    return train_loader, val_loader

## 6. Snapshot manifest

Соберём список snapshot'ов из baseline_v2 + помечаем какой метод какой prebuilder использует.

In [ ]:
import glob

SNAPSHOTS = []
for run_dir in sorted(glob.glob(f'{BASELINE_RUNS_DIR}/*/warm_start_snapshot')):
    run_name = os.path.basename(os.path.dirname(run_dir))
    if 'adalora' in run_name:
        method = 'adalora'
    elif 'l1ra' in run_name:
        method = 'l1ra'
    else:
        continue
    ds_name = run_name.split('_')[0] + ('_math' if run_name.startswith('meta') else ('_orca' if run_name.startswith('open') else ''))
    if run_name.startswith('meta_math'):
        ds_name = 'meta_math'
    elif run_name.startswith('open_orca'):
        ds_name = 'open_orca'
    SNAPSHOTS.append({
        'run_name':     run_name,
        'method':       method,
        'dataset':      ds_name,
        'snapshot_dir': run_dir,
    })
    with open(f'{run_dir}/metadata.json') as f:
        meta = json.load(f)
    print(f"{run_name:50s}  method={method:7s}  trigger_step={meta['trigger_step']:4d}  rank(mean)={meta['rank_vector_mean']:.2f}")

## 7. Source-model rebuilders

Чтобы корректно слайсить snapshot AdamW state, нужно воссоздать source-модель в той же конфигурации, что и в baseline_v2 (для ровного name → param_id mapping).

In [ ]:
def make_source_rebuilder(method: str, train_loader_len: int):
    """Фабрика исходной модели с тем же конфигом."""
    if method == 'adalora':
        total_step = (train_loader_len // GRAD_ACCUM) * NUM_EPOCHS
        def _build():
            return build_adalora_model(
                init_r=32, target_r=16,
                model_name=MODEL_NAME, total_step=total_step,
            )
        return _build
    elif method == 'l1ra':
        def _build():
            return build_l1ra_model(
                rank=32, model_name=MODEL_NAME, l1ra_lambda=1e-3,
            )
        return _build
    else:
        raise ValueError(method)

## 8. Основной цикл: warm-start -> plain LoRA

Для каждого snapshot'а:
1. Сборка plain-LoRA с rank_pattern (через `warm_start_to_plain_lora`).
2. Создать scheduler с теми же `total_steps`/`warmup_steps`, что в baseline, и загрузить старый state_dict.
3. Дотренировка через `train_model` с `start_optimizer_step=trigger_step`, `max_optimizer_steps=total_step`.

In [ ]:
b2_runs = []

for entry in SNAPSHOTS:
    run_name     = entry['run_name']
    method       = entry['method']
    ds_name      = entry['dataset']
    snapshot_dir = entry['snapshot_dir']

    out_run_name = f'{run_name}__B2_K{TARGET_K}'
    output_dir   = f'{B2_RUNS_DIR}/{out_run_name}'
    os.makedirs(output_dir, exist_ok=True)
    print(f'\n=========================  {out_run_name}  =========================')

    seed_everything(SEED)
    train_df, val_df = DATASETS[ds_name]
    train_loader, val_loader = make_loaders(train_df, val_df, seed=SEED)
    total_step = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS

    def _scheduler_factory(opt):
        return build_scheduler(opt, len(train_loader), NUM_EPOCHS, GRAD_ACCUM)

    rebuild_src = make_source_rebuilder(method, len(train_loader))

    bundle = warm_start_to_plain_lora(
        snapshot_dir       = snapshot_dir,
        method             = method,
        base_model_name    = MODEL_NAME,
        target_mean_k      = TARGET_K,
        lr                 = LR,
        weight_decay       = 0.01,
        lora_alpha         = TARGET_K * 2,
        lora_dropout       = 0.05,
        rebuild_source_model_fn = rebuild_src,
        scheduler_builder       = _scheduler_factory,
    )
    new_model     = bundle['model']
    new_optimizer = bundle['optimizer']
    new_scheduler = bundle['scheduler']
    trigger_step  = bundle['trigger_step']
    K_per_module  = bundle['K_per_module']
    K_arr = np.array(list(K_per_module.values()))
    n_kept = int((K_arr > 0).sum())
    n_drop = int((K_arr == 0).sum())

    print(f'trigger_step={trigger_step}  total_step={total_step}  remaining={total_step - trigger_step}')
    print(f'τ={bundle["calibration"]["tau"]:.4e}   K stats: '
          f'mean={K_arr.mean():.2f}  min={int(K_arr.min())}  max={int(K_arr.max())}  '
          f'kept={n_kept}/{len(K_arr)}  dropped={n_drop}')
    print(f'matched lora weights: {bundle["matched_weights"]}    '
          f'transferred optim moments: {bundle["transferred_optim"]}')
    if bundle['skipped_weights']:
        print('  skipped weights (first 3):', bundle['skipped_weights'][:3])
    if bundle['missing_optim']:
        print('  missing optim (first 3):', bundle['missing_optim'][:3])

    from collections import Counter
    applied_ranks = []
    for _n, _p in new_model.named_parameters():
        if _n.endswith('.lora_A.default.weight'):
            applied_ranks.append(_p.shape[0])
    _hist = Counter(applied_ranks)
    _mean = sum(applied_ranks) / max(len(applied_ranks), 1)
    print(f'rank_pattern check: n_modules={len(applied_ranks)}  mean_rank={_mean:.2f}  '
          f'min={min(applied_ranks)}  max={max(applied_ranks)}  '
          f'top5_hist={_hist.most_common(5)}')
    if abs(_mean - TARGET_K) > 1.5:
        print(f'!! WARNING: mean rank {_mean:.2f} far from TARGET_K={TARGET_K}; '
              'rank_pattern may not be applied correctly.')

    with open(f'{output_dir}/calibration.json', 'w') as f:
        json.dump({
            'source_run':       run_name,
            'method':           method,
            'trigger_step':     trigger_step,
            'total_step':       total_step,
            'tau':              float(bundle['calibration']['tau']),
            'target_mean_k':    TARGET_K,
            'K_mean':           float(K_arr.mean()),
            'K_min':            int(K_arr.min()),
            'K_max':            int(K_arr.max()),
            'modules_kept':     n_kept,
            'modules_dropped':  n_drop,
            'K_per_module':     {f'layer{l}.{s}': v for (l, s), v in K_per_module.items()},
        }, f, indent=2)

    skip_batches = trigger_step * GRAD_ACCUM
    print(f'will skip first {skip_batches} batches in the dataloader (= {trigger_step} optimizer steps)')

    logs = train_model(
        model                = new_model,
        train_loader         = train_loader,
        val_loader           = val_loader,
        optimizer            = new_optimizer,
        scheduler            = new_scheduler,
        output_dir           = output_dir,
        num_epochs           = NUM_EPOCHS,
        grad_accum_steps     = GRAD_ACCUM,
        device               = 'cuda',
        save_steps           = SAVE_STEPS,
        log_per_step         = LOG_PER_STEP,
        start_optimizer_step = trigger_step,
        max_optimizer_steps  = total_step,
        skip_initial_batches = skip_batches,
    )
    new_model.save_pretrained(f'{output_dir}/adapter')

    b2_runs.append({
        'source_run':   run_name,
        'method':       method,
        'dataset':      ds_name,
        'output_dir':   output_dir,
        'trigger_step': trigger_step,
        'total_step':   total_step,
        'logs':         logs,
    })

    del new_model, new_optimizer, new_scheduler
    del bundle, logs
    del train_loader, val_loader
    del rebuild_src
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        free, total = torch.cuda.mem_get_info()
        print(f'GPU free={free/2**30:.2f} GB / total={total/2**30:.2f} GB')

## 9. Results summary

In [ ]:
def best_val_ppl(logs):
    val_rows = [r for r in logs if r.get('split') == 'val' and r.get('ppl')]
    return min(r['ppl'] for r in val_rows) if val_rows else None

rows = []
for run in b2_runs:
    rows.append({
        'method':         f'B2_{run["method"]}_to_lora',
        'dataset':        run['dataset'],
        'source':         run['source_run'],
        'trigger_step':   run['trigger_step'],
        'total_step':     run['total_step'],
        'best_val_ppl':   best_val_ppl(run['logs']),
    })
summary = pd.DataFrame(rows).sort_values(['dataset', 'best_val_ppl']).reset_index(drop=True)
summary.to_csv(f'{B2_RUNS_DIR}/summary.csv', index=False)
summary

## 10. Sanity check

Каждый B2 прогон должен иметь:
- `calibration.json` (τ, K_per_module)
- `metrics.csv` с шагами от `trigger_step+1` до `total_step`
- адаптер в `adapter/`
- промежуточные чекпоинты `checkpoints/step_*` каждые 50 шагов

In [ ]:
for run_dir in sorted(glob.glob(f'{B2_RUNS_DIR}/*/')):
    name = os.path.basename(os.path.normpath(run_dir))
    ckpts = sorted(glob.glob(f'{run_dir}/checkpoints/step_*'))
    metrics = f'{run_dir}/metrics.csv'
    n_log = len(pd.read_csv(metrics)) if os.path.exists(metrics) else 0
    has_calib = os.path.exists(f'{run_dir}/calibration.json')
    print(f'{name:60s}  checkpoints={len(ckpts):3d}  rows={n_log}  calib={has_calib}')